# SBC Validation: RJMCMC Model Selection (Sinusoids)

Simulation-Based Calibration for reversible-jump MCMC model selection.

Tests whether the sampler correctly recovers:
1. **Model index** — via randomized PIT of the true nmodel among posterior model probs
2. **Continuous source parameters** — conditional SBC ranks for simulations where the true model is nmodel=0

**Setup:**
- MAX_SOURCES = 3, NUM_PARAMS = 3 (amplitude, frequency, phase)
- Model prior: uniform on {0, 1, 2}
- Source prior: A ~ U(0.5, 5), f ~ U(0.1, 3), φ ~ U(0, π)
- Data: 50 time points, noise σ = 0.5
- Sampler: `PTSampler.from_rjmcmc(space, ntemps=8)`, 10000 iterations

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

from impulse import (
    PTSampler,
    BirthDeathProductSpace,
    run_sbc_model_selection,
    compute_sbc_quantile,
    sbc_ecdf_plot,
    ecdf,
)

plt.rcParams.update({"figure.dpi": 120})

## 1. Define Sinusoid Model

In [ ]:
MAX_SOURCES = 3
NUM_PARAMS = 3  # amplitude, frequency, phase
NUM_MODELS = MAX_SOURCES  # nmodel in {0, 1, 2}
N_TIME = 50
SIGMA = 0.5

# Prior bounds per source
A_MIN, A_MAX = 0.5, 5.0
F_MIN, F_MAX = 0.1, 3.0
PHI_MIN, PHI_MAX = 0.0, np.pi

T_GRID = np.linspace(0, 2 * np.pi, N_TIME)


class SinLikelihood:
    """Log-likelihood for sum-of-sinusoids model."""
    def __init__(self, t, data, sigma):
        self.t = t
        self.data = data
        self.sigma = sigma

    def __call__(self, params):
        n_src = len(params) // NUM_PARAMS
        model = np.zeros_like(self.t)
        for k in range(n_src):
            a = params[k * NUM_PARAMS]
            f = params[k * NUM_PARAMS + 1]
            phi = params[k * NUM_PARAMS + 2]
            model += a * np.sin(f * self.t + phi)
        r = self.data - model
        return -0.5 * np.sum((r / self.sigma) ** 2)


class SinPrior:
    """Uniform prior on all source params (active + inactive)."""
    def __call__(self, params):
        n_src = len(params) // NUM_PARAMS
        for k in range(n_src):
            a = params[k * NUM_PARAMS]
            f = params[k * NUM_PARAMS + 1]
            phi = params[k * NUM_PARAMS + 2]
            if not (A_MIN <= a <= A_MAX):
                return -np.inf
            if not (F_MIN <= f <= F_MAX):
                return -np.inf
            if not (PHI_MIN <= phi <= PHI_MAX):
                return -np.inf
        return 0.0


def source_prior_draw(rng):
    """Draw one source's params from the prior."""
    return np.array([
        rng.uniform(A_MIN, A_MAX),
        rng.uniform(F_MIN, F_MAX),
        rng.uniform(PHI_MIN, PHI_MAX),
    ])

## 2. Prior Draws and Data Generation

In [ ]:
def model_prior_draw(rng):
    """Uniform prior on nmodel in {0, 1, 2}."""
    return rng.integers(0, NUM_MODELS)


def param_prior_draw(nmodel, rng):
    """Draw params for (nmodel+1) active sources."""
    n_active = nmodel + 1
    params = np.concatenate([source_prior_draw(rng) for _ in range(n_active)])
    return params


def data_generator(true_nmodel, true_params, rng):
    """Generate noisy sinusoidal data."""
    n_src = true_nmodel + 1
    model = np.zeros_like(T_GRID)
    for k in range(n_src):
        a = true_params[k * NUM_PARAMS]
        f = true_params[k * NUM_PARAMS + 1]
        phi = true_params[k * NUM_PARAMS + 2]
        model += a * np.sin(f * T_GRID + phi)
    return model + rng.normal(0, SIGMA, size=N_TIME)

## 3. Sampler Factory

In [ ]:
def rjmcmc_factory(true_nmodel, true_params, data, rng, outdir):
    """Build and run an RJMCMC sampler."""
    lnlike = SinLikelihood(T_GRID, data, SIGMA)
    lnprior = SinPrior()

    space = BirthDeathProductSpace(
        loglikelihood=lnlike,
        logprior=lnprior,
        num_sources=MAX_SOURCES,
        num_params=NUM_PARAMS,
        source_prior_draw=source_prior_draw,
    )

    sampler = PTSampler.from_rjmcmc(
        space,
        ntemps=8,
        seed=int(rng.integers(0, 2**31)),
        outdir=outdir,
        save_freq=10000,
    )

    x0 = space.draw_initial_position(rng, nmodel=0)
    sampler.sample(x0, num_iterations=10000)
    return sampler, space

## 4. Run SBC Model Selection (200 simulations)

In [ ]:
N_SIM = 200
BURN = 3000
SEED = 123

rjmcmc_results = run_sbc_model_selection(
    sampler_factory=rjmcmc_factory,
    model_prior_draw=model_prior_draw,
    param_prior_draw=param_prior_draw,
    data_generator=data_generator,
    num_models=NUM_MODELS,
    n_simulations=N_SIM,
    burn=BURN,
    seed=SEED,
)

## 5. Figure 1: Model Index PIT ECDF

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

pit = rjmcmc_results["pit_values"]
sbc_ecdf_plot(pit[:, None], param_names=["Model index PIT"], ax=ax)
ax.set_title("RJMCMC Model Index — PIT ECDF")

fig.tight_layout()
plt.show()

## 6. Figure 2: Model Index PIT Histogram

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

ax.hist(pit, bins=20, range=(0, 1), edgecolor="black", alpha=0.7)
ax.axhline(N_SIM / 20, color="red", ls="--", lw=1, label="expected")
ax.set_xlabel("PIT value")
ax.set_ylabel("Count")
ax.set_title("Model Index PIT Histogram")
ax.legend()

fig.tight_layout()
plt.show()

## 7. Figure 3: Conditional Continuous Param ECDF (nmodel=0 subset)

For simulations where the true model is nmodel=0 (single source, no label switching),
compute SBC ranks of the 3 source params among posterior samples filtered to nmodel=0.

In [ ]:
# This requires re-running the loop for the nmodel=0 subset
# to extract chain samples. We do a lightweight re-analysis.
import tempfile, shutil
from tqdm import tqdm

rng = np.random.default_rng(SEED)
n_cond = 80  # fewer sims for the conditional analysis
cond_quantiles = []

for i in tqdm(range(n_cond), desc="Conditional SBC (nmodel=0)"):
    true_nmodel = 0
    true_params = param_prior_draw(true_nmodel, rng)
    data = data_generator(true_nmodel, true_params, rng)

    tmpdir = tempfile.mkdtemp()
    try:
        sim_rng = np.random.default_rng(rng.integers(0, 2**32))
        sampler, space = rjmcmc_factory(true_nmodel, true_params, data, sim_rng, tmpdir)
        chain = sampler.load_chain()["samples"][0]  # cold chain
        chain = chain[BURN:]

        # Filter to nmodel=0
        nmodel_col = np.rint(chain[:, -1]).astype(int)
        mask = nmodel_col == 0
        if mask.sum() < 20:
            continue  # skip if too few samples in this model

        # First source params: columns 0:NUM_PARAMS
        filtered = chain[mask, :NUM_PARAMS]
        q = compute_sbc_quantile(true_params[:NUM_PARAMS], filtered)
        cond_quantiles.append(q)
    finally:
        shutil.rmtree(tmpdir, ignore_errors=True)

cond_quantiles = np.array(cond_quantiles)
print(f"Usable simulations: {len(cond_quantiles)} / {n_cond}")

In [ ]:
if len(cond_quantiles) > 10:
    fig, ax = plt.subplots(figsize=(6, 5))
    sbc_ecdf_plot(
        cond_quantiles,
        param_names=["Amplitude", "Frequency", "Phase"],
        ax=ax,
    )
    ax.set_title("Conditional SBC ECDF (nmodel=0)")
    fig.tight_layout()
    plt.show()
else:
    print("Too few usable simulations for conditional ECDF plot.")

## 8. Summary

In [ ]:
# Model index PIT
stat, pval = stats.kstest(pit, "uniform")
print(f"Model Index PIT:  KS stat = {stat:.4f}, p-value = {pval:.4f}")

# Model recovery rates
true_models = rjmcmc_results["true_models"]
post_probs = rjmcmc_results["posterior_probs"]
map_models = np.argmax(post_probs, axis=1)
recovery_rate = np.mean(map_models == true_models)
print(f"MAP model recovery rate: {recovery_rate:.1%}")

# Per-model recovery
for k in range(NUM_MODELS):
    mask = true_models == k
    if mask.sum() > 0:
        rate = np.mean(map_models[mask] == k)
        print(f"  nmodel={k}: {rate:.1%} ({mask.sum()} sims)")

# Conditional continuous params
if len(cond_quantiles) > 10:
    print("\nConditional continuous params (nmodel=0):")
    for j, name in enumerate(["Amplitude", "Frequency", "Phase"]):
        stat, pval = stats.kstest(cond_quantiles[:, j], "uniform")
        flag = "" if pval > 0.05 else " ***"
        print(f"  {name}: KS stat = {stat:.4f}, p-value = {pval:.4f}{flag}")